In [43]:


import numpy
import os
import pandas as pd 

## 1. Processing customer data from a CSV file for a retail company to ensure clean, unique records.
### Tasks involved: Deduplication, Cleaning

In [44]:

transaction_data = pd.read_csv('transaction_data.csv')
customer_data = pd.read_csv('us_customer_data.csv')

In [45]:
print('transaction_data count',transaction_data.shape)
print('customer_data count', customer_data.shape)


transaction_data count (1000, 7)
customer_data count (1000, 7)


In [46]:


def describe_data(df,head = 5):
    print('datatypes')
    print(df.dtypes.value_counts())

    print('missing values')
    print(df.isnull().sum())

    print('duplicated values')
    print(df.duplicated().sum())

    print('data statistics')
    print(df.info())

describe_data(transaction_data)
describe_data(customer_data)

datatypes
object     4
int64      2
float64    1
Name: count, dtype: int64
missing values
transaction_id       0
customer_id          0
amount              50
transaction_date     0
product_category     0
payment_method       0
store_location       0
dtype: int64
duplicated values
0
data statistics
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    1000 non-null   int64  
 1   customer_id       1000 non-null   int64  
 2   amount            950 non-null    float64
 3   transaction_date  1000 non-null   object 
 4   product_category  1000 non-null   object 
 5   payment_method    1000 non-null   object 
 6   store_location    1000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 54.8+ KB
None
datatypes
object    6
int64     1
Name: count, dtype: int64
missing values
customer_id           0
name 

In [47]:
print("Missing customer data in each column after cleaning customerID :\n",customer_data.isnull().sum())
customer_data.dropna(subset=["customer_id"],axis = 0 , inplace = True)



Missing customer data in each column after cleaning customerID :
 customer_id           0
name                  0
email                50
phone                50
address               0
registration_date     0
loyalty_status        0
dtype: int64


In [48]:
print("Missing customer data in each column after cleaning customerID :\n",customer_data.isnull().sum())



Missing customer data in each column after cleaning customerID :
 customer_id           0
name                  0
email                50
phone                50
address               0
registration_date     0
loyalty_status        0
dtype: int64


In [49]:
print('before cleaning' , customer_data.email.duplicated().sum())
customer_data = customer_data.drop_duplicates(keep='first')
print('after cleaning' , customer_data.duplicated().sum())

before cleaning 139
after cleaning 0


In [50]:
customer_data.shape

(1000, 7)

## 2. Standardizing contact details in an Excel file for a marketing campaign using  customer data
### Tasks involved: Cleaning, Mapping

In [51]:
# remove whitespace 
import pandas as pd
import re

def clean_spaces(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())


In [52]:

for col in ['email','phone','name']:
    if customer_data[col].dtype == "object":
     customer_data[col] = customer_data[col].apply(clean_spaces)


In [53]:
def proper_case_name(name):
    name = clean_spaces(name)
    return name.title() if name else ""

customer_data["name"] = customer_data["name"].apply(proper_case_name)


In [54]:
# fix email error
EMAIL_REGEX = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"

def clean_email(email):
    if email is None:
        return ""
    email = clean_spaces(email).lower()
    return email

def email_valid(email):
    return bool(re.match(EMAIL_REGEX, email)) if email else False

customer_data["email"] = customer_data["email"].apply(clean_email)
# Optional: blank out invalid emails


In [55]:
# standardize phone number
import re
# formats
# (716)547-9134x123
# 001-716-547-9134x123
# 716.547.9134x123
# rest are junk
# output: us standar: phoneno: +1-716- 547-9134 phoneext: digits if present


s = str('(716)547-9134').strip()




In [56]:
import pandas as pd
import re

def clean_us_phone_with_ext(raw):
    """
    Returns (phone_e164, ext) for US numbers.
    phone_e164 format: +1-AAA-BBB-CCCC
    ext is digits only (string) or None
    """
    if pd.isna(raw):
        return (None, None)

    s = str(raw).strip()
    if s == "":
        return (None, None)

    # --- 1) Extract extension (x123, ext 123, extension 123) ---
    ext = None
    m = re.search(r'(?:ext\.?|extension|x)\s*(\d+)\s*$', s, flags=re.IGNORECASE)
    if m:
        ext = m.group(1)
        s = s[:m.start()].strip()  # remove extension part from main number

    # --- 2) Keep digits only from the remaining number ---
    digits = re.sub(r"\D", "", s)

    # --- 3) Remove leading international prefix "001" (common in your data) ---
    if digits.startswith("001"):
        digits = digits[3:]

    # --- 4) Normalize to US 10-digit ---
    # If 11 digits and starts with 1 -> drop country code
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]

    # If not exactly 10 digits now, it's invalid for US standardization
    if len(digits) != 10:
        return (None, ext)  # keep ext if it existed, but number invalid

    phone_e164 = f"+1-{digits[:3]}-{digits[3:6]}-{digits[6:]}"
    return (phone_e164, ext)

# Apply to your dataframe
customer_data[["phone_e164", "phone_ext"]] = customer_data["phone"].apply(
    lambda x: pd.Series(clean_us_phone_with_ext(x))
)

# (Optional) Replace original phone with standardized phone
customer_data["phone"] = customer_data["phone_e164"]
customer_data.drop(columns=["phone_e164"], inplace=True)

customer_data.head()


,customer_id,name,email,phone,address,registration_date,loyalty_status,phone_ext
0,1,Michelle Kidd,vayala@example.net,None,"USNS Santiago, FPO AE 80872",2025-01-25,Gold,None
1,2,Brad Newton,taylorcatherine@example.net,+1-759-518-8536,"38783 Oliver Street, West Kristenborough, MT 9...",2023-07-13,Silver,738
2,3,Larry Torres,dsanchez@example.net,+1-323-525-3094,"6845 Steele Turnpike, West Erikabury, UT 37487",2023-08-18,Bronze,96062
3,4,Kimberly Price,jessicaknight@example.com,+1-947-633-4224,"1631 Alexis Meadows, Lake Amanda, CA 75179",2024-12-08,Gold,07930
4,5,Matthew Phillips,qwilliams@example.com,+1-869-650-5682,"2274 Williams Heights Suite 895, Andersonhaven...",2024-02-03,Gold,8385


In [57]:
transaction_data.head()

,transaction_id,customer_id,amount,transaction_date,product_category,payment_method,store_location
0,1,565,2992.47,2025-03-10 01:20:54,Sports,Debit Card,New York
1,2,323,2041.87,2025-01-02 15:24:19,Clothing,Cash,New York
2,3,398,107.35,2025-02-16 03:49:01,Beauty,Debit Card,Online
3,4,19,NaN,2025-04-30 15:26:23,Sports,Debit Card,Los Angeles
4,5,547,3063.28,2025-06-14 04:28:53,Clothing,PayPal,Los Angeles


In [58]:
print(transaction_data.info())
print(transaction_data.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    1000 non-null   int64  
 1   customer_id       1000 non-null   int64  
 2   amount            950 non-null    float64
 3   transaction_date  1000 non-null   object 
 4   product_category  1000 non-null   object 
 5   payment_method    1000 non-null   object 
 6   store_location    1000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 54.8+ KB
None
transaction_id       0
customer_id          0
amount              50
transaction_date     0
product_category     0
payment_method       0
store_location       0
dtype: int64


In [59]:
transaction_data = transaction_data.dropna(subset=["amount"])


In [60]:
transaction_data["amount"] = pd.to_numeric(
    transaction_data["amount"], errors="coerce"
)


In [61]:
transaction_data["transaction_date"] = pd.to_datetime(
    transaction_data["transaction_date"], errors="coerce"
)


In [62]:
print(transaction_data.info())
print(transaction_data.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 950 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    950 non-null    int64         
 1   customer_id       950 non-null    int64         
 2   amount            950 non-null    float64       
 3   transaction_date  950 non-null    datetime64[ns]
 4   product_category  950 non-null    object        
 5   payment_method    950 non-null    object        
 6   store_location    950 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 59.4+ KB
None
transaction_id      0
customer_id         0
amount              0
transaction_date    0
product_category    0
payment_method      0
store_location      0
dtype: int64


In [63]:
transaction_data["amount"] = pd.to_numeric(transaction_data["amount"], errors="coerce")

# --- 3) Compute mean (NaN ignored automatically) ---
mean_amount = transaction_data["amount"].mean()

# --- 4) Flags (methods) ---
transaction_data["is_high_value_1000"] = transaction_data["amount"].gt(1000)
transaction_data["is_high_value_above_mean"] = transaction_data["amount"].gt(mean_amount)

# --- 5) Promotion tier: Gold / Silver / Bronze ---
def assign_tier(row):
    if row["is_high_value_above_mean"]:
        return "Gold"
    elif row["is_high_value_1000"]:
        return "Silver"
    else:
        return "Bronze"

transaction_data["promotion_tier"] = transaction_data.apply(assign_tier, axis=1)

# --- 6) (Optional but recommended) Clean date column ---
transaction_data["transaction_date"] = pd.to_datetime(
    transaction_data["transaction_date"], errors="coerce"
)

# --- 7) Quick checks ---
print("Mean amount:", mean_amount)
print(transaction_data["promotion_tier"].value_counts(dropna=False))

# Preview
transaction_data.head()


Mean amount: 2532.5501263157894
promotion_tier
Gold      482
Silver    288
Bronze    180
Name: count, dtype: int64


,transaction_id,customer_id,amount,transaction_date,product_category,payment_method,store_location,is_high_value_1000,is_high_value_above_mean,promotion_tier
0,1,565,2992.47,2025-03-10 01:20:54,Sports,Debit Card,New York,True,True,Gold
1,2,323,2041.87,2025-01-02 15:24:19,Clothing,Cash,New York,True,False,Silver
2,3,398,107.35,2025-02-16 03:49:01,Beauty,Debit Card,Online,False,False,Bronze
4,5,547,3063.28,2025-06-14 04:28:53,Clothing,PayPal,Los Angeles,True,True,Gold
5,6,752,441.49,2025-03-28 06:15:55,Electronics,Cash,New York,False,False,Bronze


## import df into sql server

In [64]:
import pyodbc

DRIVER_PATH = "/opt/homebrew/lib/libmsodbcsql.18.dylib"  # <-- change this

conn_str = (
    f"DRIVER={{{DRIVER_PATH}}};"
    "SERVER=localhost;"
    "DATABASE=marketingdb;"
    "UID=sa;"
    "PWD=Suman@2026!SQL;"
    "Encrypt=no;"
)

cnxn = pyodbc.connect(conn_str)
cur = cnxn.cursor()
cur.execute("SELECT TOP 5 * FROM dbo.customer_data;")
cur.fetchall()


[(1, 'Michelle Kidd', 'vayala@example.net', None, 'USNS Santiago, FPO AE 80872', '2025-01-25', 'Gold', None),
 (2, 'Brad Newton', 'taylorcatherine@example.net', '+1-759-518-8536', '38783 Oliver Street, West Kristenborough, MT 99752', '2023-07-13', 'Silver', '738'),
 (3, 'Larry Torres', 'dsanchez@example.net', '+1-323-525-3094', '6845 Steele Turnpike, West Erikabury, UT 37487', '2023-08-18', 'Bronze', '96062'),
 (4, 'Kimberly Price', 'jessicaknight@example.com', '+1-947-633-4224', '1631 Alexis Meadows, Lake Amanda, CA 75179', '2024-12-08', 'Gold', '07930'),
 (5, 'Matthew Phillips', 'qwilliams@example.com', '+1-869-650-5682', '2274 Williams Heights Suite 895, Andersonhaven, OR 80565', '2024-02-03', 'Gold', '8385')]

In [65]:

from sqlalchemy import create_engine
from urllib.parse import quote_plus

USER = "sa"
PASSWORD = quote_plus("Suman@2026!SQL")  # URL-encode special chars
SERVER = "localhost"
DB = "marketingdb"

DRIVER_PATH = "/opt/homebrew/lib/libmsodbcsql.18.dylib"

connection_string = (
    f"mssql+pyodbc://{USER}:{PASSWORD}@{SERVER}/{DB}"
    f"?driver={quote_plus(DRIVER_PATH)}"
    "&Encrypt=no"
)

engine = create_engine(connection_string)


customer_df = customer_data.copy()
transaction_df = transaction_data.copy()



transaction_df.to_sql(
    name="transaction_data",
    con=engine,
    if_exists="replace",
    index=False
)

customer_df.to_sql(
    name="customer_data",
    con=engine,
    if_exists="replace",
    index=False
)


214

In [66]:
df = pd.read_sql("select * from customer_data",engine)
df.head()

,customer_id,name,email,phone,address,registration_date,loyalty_status,phone_ext
0,1,Michelle Kidd,vayala@example.net,None,"USNS Santiago, FPO AE 80872",2025-01-25,Gold,None
1,2,Brad Newton,taylorcatherine@example.net,+1-759-518-8536,"38783 Oliver Street, West Kristenborough, MT 9...",2023-07-13,Silver,738
2,3,Larry Torres,dsanchez@example.net,+1-323-525-3094,"6845 Steele Turnpike, West Erikabury, UT 37487",2023-08-18,Bronze,96062
3,4,Kimberly Price,jessicaknight@example.com,+1-947-633-4224,"1631 Alexis Meadows, Lake Amanda, CA 75179",2024-12-08,Gold,07930
4,5,Matthew Phillips,qwilliams@example.com,+1-869-650-5682,"2274 Williams Heights Suite 895, Andersonhaven...",2024-02-03,Gold,8385


In [67]:
transaction_df.head()

,transaction_id,customer_id,amount,transaction_date,product_category,payment_method,store_location,is_high_value_1000,is_high_value_above_mean,promotion_tier
0,1,565,2992.47,2025-03-10 01:20:54,Sports,Debit Card,New York,True,True,Gold
1,2,323,2041.87,2025-01-02 15:24:19,Clothing,Cash,New York,True,False,Silver
2,3,398,107.35,2025-02-16 03:49:01,Beauty,Debit Card,Online,False,False,Bronze
4,5,547,3063.28,2025-06-14 04:28:53,Clothing,PayPal,Los Angeles,True,True,Gold
5,6,752,441.49,2025-03-28 06:15:55,Electronics,Cash,New York,False,False,Bronze


# order csv handling


In [68]:
orders = pd.read_csv('order_data.csv')

In [69]:
orders.head()

,order_id,customer_id,order_date,order_amount,order_status,product_category
0,892a07a4-d252-4775-85e0-73077143e1c6,966,2024-11-27,317.64,Cancelled,Home & Garden
1,ae160758-e187-47b2-9350-032f88f55491,345,2023-03-27,645.87,Completed,Home & Garden
2,7c50456e-6123-45cc-aa19-128bef3754d6,503,2024-03-31,880.86,Pending,Clothing
3,c788b56b-3716-4cd9-a827-d4dc401ba00c,385,2023-08-09,876.83,Cancelled,Home & Garden
4,925ab5b1-adb1-4302-a70b-1c2db724e02b,817,2023-04-19,264.53,Pending,Home & Garden


In [70]:
print(orders.info())
print(orders.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_id          2000 non-null   object 
 1   customer_id       2000 non-null   int64  
 2   order_date        2000 non-null   object 
 3   order_amount      2000 non-null   float64
 4   order_status      2000 non-null   object 
 5   product_category  2000 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 93.9+ KB
None
order_id            0
customer_id         0
order_date          0
order_amount        0
order_status        0
product_category    0
dtype: int64


In [71]:
order_df = orders.copy()
order_df.to_sql(
    name = 'orders',
    con = engine,
    if_exists= 'replace',
    index=False
)

255

# 4	SQL Server	Extracting customer information from a SQL Server database for reporting purposes.	Extraction, Mapping	Beginner		

In [72]:
query = """ 
select  c.customer_id,
    c.name           AS customer_name,
    c.email,
    c.phone,
    c.loyalty_status,
    c.address,
    o.order_id,
    o.order_date,
    o.order_amount,
    o.order_status
from customer_data c 
left join orders o on c.customer_id= o.customer_id

"""
df = pd.read_sql(query,engine)


# Load the Unified Customer View back into SQL Server for further analysis and reporting.

df.to_sql(
    name = 'customer_orders',
    con = engine,
    if_exists= 'replace',
    index=False
)


42

In [73]:
import re
import pandas as pd

df = df.copy()

def split_phone(phone):
    if pd.isna(phone):
        return pd.NA, pd.NA
    
    s = str(phone).strip()
    
    # Extract country code from +<digits>
    m = re.match(r"^\+(\d{1,3})", s)
    digits = re.sub(r"\D", "", s)  # all digits
    
    if not m:
        # no +countrycode present
        return pd.NA, digits
    
    cc = m.group(1)  # e.g., "1", "91", "44"
    
    # remove ONLY that cc if digits starts with it
    if digits.startswith(cc):
        national = digits[len(cc):]
    else:
        national = digits
    
    return cc, national

df[["dialing_code", "phone_digits"]] = df["phone"].apply(split_phone).apply(pd.Series)



In [74]:
# create two columns
# remove prefixes mr mrs miss dr,
# remove suffixes jr sr II,III

prefixes = r"^(mr|mrs|miss|dr)\.?\s+"
suffixes = r"\s+(jr|sr|ii|iii)\.?$"


In [75]:
df =df.copy()

# convert to string 
# 
df["customer_name"] = df["customer_name"].astype(str)

# remove prefixes 
df["customer_name"] = (df["customer_name"].str.lower()
    .str.replace(prefixes,"",regex = True)
    .str.replace(suffixes,"",regex = True)
    .str.title()
)

In [76]:
df[["first_name","last_name"]] = (
    df["customer_name"].str.split(" ",n=1,expand=True)
)


In [77]:
df.columns


Index(['customer_id', 'customer_name', 'email', 'phone', 'loyalty_status',
       'address', 'order_id', 'order_date', 'order_amount', 'order_status',
       'dialing_code', 'phone_digits', 'first_name', 'last_name'],
      dtype='object')

In [78]:
df["country_code "] = df["address"].str.extract(r",\s*([A-Z]{2})\s+\d{5}",expand= False)

In [79]:
df.head()

,customer_id,customer_name,email,phone,loyalty_status,address,order_id,order_date,order_amount,order_status,dialing_code,phone_digits,first_name,last_name,country_code
0,1,Michelle Kidd,vayala@example.net,None,Gold,"USNS Santiago, FPO AE 80872",None,None,NaN,None,<NA>,<NA>,Michelle,Kidd,NaN
1,2,Brad Newton,taylorcatherine@example.net,+1-759-518-8536,Silver,"38783 Oliver Street, West Kristenborough, MT 9...",209c024e-6c32-49fa-acd1-2501bd19c1cd,2023-10-30,486.53,Pending,1,7595188536,Brad,Newton,MT
2,2,Brad Newton,taylorcatherine@example.net,+1-759-518-8536,Silver,"38783 Oliver Street, West Kristenborough, MT 9...",3f263091-da18-48a6-8b00-c179e1b1a81a,2024-08-29,781.22,Pending,1,7595188536,Brad,Newton,MT
3,3,Larry Torres,dsanchez@example.net,+1-323-525-3094,Bronze,"6845 Steele Turnpike, West Erikabury, UT 37487",63d7870e-617e-4f49-ae9b-b893e635e43f,2023-04-05,454.07,Pending,1,3235253094,Larry,Torres,UT
4,4,Kimberly Price,jessicaknight@example.com,+1-947-633-4224,Gold,"1631 Alexis Meadows, Lake Amanda, CA 75179",21209e05-078a-402d-a450-dd310595e5f0,2024-08-27,117.03,Completed,1,9476334224,Kimberly,Price,CA


In [80]:
tier_map = {
    "Gold": 2,
    "Silver": 1,
    "Bronze": 0
}

df["Customer_Tier"] = df["loyalty_status"].map(tier_map)


In [84]:
df = df[
    [
        # Customer identity
        "customer_id",
        "first_name",
        "last_name",
        "customer_name",

        # Contact information
        "email",
        "phone",
        "dialing_code",
        "phone_digits",

        # Geography
        "address",
        "country_code ",

        # Classification
        "loyalty_status",
        "Customer_Tier",

        # Order information
        "order_id",
        "order_date",
        "order_amount",
        "order_status",
    ]
]


In [85]:
df.to_sql(
    name = 'customer_orders',
    con = engine,
    if_exists= 'replace',
    index=False
)


36

# clean and add product_inventory